# ViT CPU vs GPU Analysis
Averages triplicates (rep 0, 1, 2) and computes standard deviation across runs.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT')
OUTPUT_DIR = Path('/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns that identify a unique experimental condition (grouping keys)
GROUP_COLS = ['model_type', 'dataset', 'model_name', 'pruning_method',
              'stored_precision', 'architecture', 'device']

# Numeric measurement columns to average and compute std over triplicates
METRIC_COLS = [
    'mean_latency_ms', 'std_latency_ms', 'median_latency_ms',
    'p25_latency_ms', 'p75_latency_ms', 'p90_latency_ms',
    'min_latency_ms', 'max_latency_ms', 'throughput_imgs_per_s',
    'mean_energy_kwh_per_image', 'mean_cpu_power_w', 'mean_gpu_power_w',
    'mean_ram_power_w'
]

# Columns that are constant across reps -- just take first value
CONST_COLS = ['num_params', 'model_size_mb', 'num_images']

datasets = ['bloodmnist', 'chestmnist', 'dermamnist', 'pathmnist']
all_dfs = []

for dataset in datasets:
    csv_path = DATA_DIR / f'{dataset}_results.csv'
    df = pd.read_csv(csv_path)
    all_dfs.append(df)
    print(f'Loaded {csv_path.name}: {len(df)} rows')

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\nCombined: {len(combined)} rows across {combined["dataset"].nunique()} datasets')
print(f'Reps per condition check -- unique rep values: {sorted(combined["rep"].unique())}')

# Compute mean and std across triplicates
agg_mean = combined.groupby(GROUP_COLS)[METRIC_COLS].mean().reset_index()
agg_std  = combined.groupby(GROUP_COLS)[METRIC_COLS].std(ddof=1).reset_index()

# Rename: original name -> _mean and _std suffixes
agg_mean = agg_mean.rename(columns={c: f'{c}_mean' for c in METRIC_COLS})
agg_std  = agg_std.rename(columns={c: f'{c}_std'  for c in METRIC_COLS})

# Constant columns: take first rep value
const_vals = combined.groupby(GROUP_COLS)[CONST_COLS].first().reset_index()

# Merge everything
result = agg_mean.merge(agg_std, on=GROUP_COLS).merge(const_vals, on=GROUP_COLS)

# Reorder: group cols | const cols | interleaved mean/std pairs
interleaved = [col for metric in METRIC_COLS for col in (f'{metric}_mean', f'{metric}_std')]
result = result[GROUP_COLS + CONST_COLS + interleaved]

print(f'Output shape: {result.shape}')
result.head()

# Save
out_path = OUTPUT_DIR / 'ViT_averaged.csv'
result.to_csv(out_path, index=False)
print(f'Saved to {out_path}')
result

Loaded bloodmnist_results.csv: 54 rows
Loaded chestmnist_results.csv: 54 rows
Loaded dermamnist_results.csv: 54 rows
Loaded pathmnist_results.csv: 54 rows

Combined: 216 rows across 4 datasets
Reps per condition check -- unique rep values: [0, 1, 2]
Output shape: (72, 36)
Saved to /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/ViT_averaged.csv


,model_type,dataset,model_name,pruning_method,stored_precision,architecture,device,num_params,model_size_mb,num_images,...,throughput_imgs_per_s_mean,throughput_imgs_per_s_std,mean_energy_kwh_per_image_mean,mean_energy_kwh_per_image_std,mean_cpu_power_w_mean,mean_cpu_power_w_std,mean_gpu_power_w_mean,mean_gpu_power_w_std,mean_ram_power_w_mean,mean_ram_power_w_std
0,ViT,bloodmnist,kd_vit_base_patch16_224_to_vit_small_patch16_2...,kd,fp32,vit_small_patch16_224,cpu,21668744,260.248,10,...,10.152282,0.035800,4.788235e-07,1.687298e-09,17.5,0.0,NaN,NaN,NaN,NaN
1,ViT,bloodmnist,kd_vit_base_patch16_224_to_vit_small_patch16_2...,kd,fp32,vit_small_patch16_224,cuda,21668744,260.248,10,...,212.740108,1.603654,1.133705e-07,4.561465e-10,17.5,0.0,69.324727,0.322243,NaN,NaN
2,ViT,bloodmnist,kd_vit_base_patch16_224_to_vit_tiny_patch16_22...,kd,fp32,vit_tiny_patch16_224,cpu,5525960,66.533,10,...,29.314553,0.235963,1.658331e-07,1.339664e-09,17.5,0.0,NaN,NaN,NaN,NaN
3,ViT,bloodmnist,kd_vit_base_patch16_224_to_vit_tiny_patch16_22...,kd,fp32,vit_tiny_patch16_224,cuda,5525960,66.533,10,...,221.653387,1.319059,8.319369e-08,4.809005e-10,17.5,0.0,48.883209,0.170742,NaN,NaN
4,ViT,bloodmnist,kd_vit_small_patch16_224_to_vit_tiny_patch16_2...,kd,fp32,vit_tiny_patch16_224,cpu,5525960,66.534,10,...,29.387196,0.051533,1.654163e-07,2.900501e-10,17.5,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ViT,pathmnist,vit_base_patch16_224_pathmnist_pretrained,baseline,fp32,vit_base_patch16_224,cuda,85805577,1029.875,10,...,80.085074,0.277927,3.030380e-07,2.998049e-10,17.5,0.0,69.867550,0.216676,NaN,NaN
68,ViT,pathmnist,vit_small_patch16_224_pathmnist_pretrained,baseline,fp32,vit_small_patch16_224,cpu,21669129,260.238,10,...,10.130843,0.002214,4.798328e-07,1.048817e-10,17.5,0.0,NaN,NaN,NaN,NaN
69,ViT,pathmnist,vit_small_patch16_224_pathmnist_pretrained,baseline,fp32,vit_small_patch16_224,cuda,21669129,260.238,10,...,212.831273,0.544478,1.127312e-07,4.649791e-10,17.5,0.0,68.874243,0.544874,NaN,NaN
70,ViT,pathmnist,vit_tiny_patch16_224_pathmnist_pretrained,baseline,fp32,vit_tiny_patch16_224,cpu,5526153,66.521,10,...,29.126661,0.025526,1.668957e-07,1.463208e-10,17.5,0.0,NaN,NaN,NaN,NaN


In [2]:
"""
vit_visualize.py

Generates charts and summary tables from ViT_averaged.csv.

Chart encoding:
    x-axis   : compression method  (baseline | oneshot | quantization | kd)
    color    : device               (CPU = green, GPU = red)
    marker   : stored precision     (fp32 = triangle ▲, fp16 = circle ●)
    size     : architecture         (tiny = small, small = medium, base = large)
    panels   : one column per dataset (bloodmnist | chestmnist | dermamnist | pathmnist)
"""

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

METHOD_LABELS = {
    "baseline":     "Baseline",
    "oneshot":      "Structured\nPruning",
    "kd":           "Knowledge\nDistillation",
}

METHOD_ORDER = ["baseline", "oneshot", "kd"]

ARCH_SIZES = {
    "vit_tiny_patch16_224":  50,
    "vit_small_patch16_224": 110,
    "vit_base_patch16_224":  200,
}
ARCH_LABELS = {
    "vit_tiny_patch16_224":  "ViT-Tiny",
    "vit_small_patch16_224": "ViT-Small",
    "vit_base_patch16_224":  "ViT-Base",
}
# Jitter offsets per architecture so same-method points don't overlap
ARCH_JITTER = {
    "vit_tiny_patch16_224":  -0.18,
    "vit_small_patch16_224":  0.0,
    "vit_base_patch16_224":   0.18,
}

DATASETS = ["bloodmnist", "chestmnist", "dermamnist", "pathmnist"]
DATASET_LABELS = {
    "bloodmnist": "BloodMNIST",
    "chestmnist": "ChestMNIST",
    "dermamnist": "DermaMNIST",
    "pathmnist":  "PathMNIST",
}

DEVICE_COLORS  = {"cpu": "#2e7d32", "cuda": "#c62828"}
PRECISION_MARKERS = {"fp32": "^", "fp16": "o"}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.dpi": 150,
})

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    if "model_type" in df.columns:
        df = df[df["model_type"] == "ViT"].copy()

    # Normalise pruning_method: the raw column has values like "baseline",
    # "oneshot", "quantization", "kd". Ensure consistent lower-case.
    df["pruning_method"] = df["pruning_method"].str.strip().str.lower()

    # Drop quantization rows — excluded from ViT analysis
    df = df[df["pruning_method"] != "quantization"].copy()

    # Normalise architecture: infer from model_name if architecture column is missing/null
    if "architecture" not in df.columns or df["architecture"].isna().all():
        df["architecture"] = df["model_name"].apply(_infer_arch)
    else:
        df["architecture"] = df["architecture"].fillna(
            df["model_name"].apply(_infer_arch)
        )

    df["method_order"] = df["pruning_method"].map(
        {m: i for i, m in enumerate(METHOD_ORDER)}
    ).fillna(99)

    return df


def _infer_arch(model_name: str) -> str:
    for arch in ["vit_base_patch16_224", "vit_small_patch16_224", "vit_tiny_patch16_224"]:
        if arch in str(model_name):
            return arch
    return "vit_tiny_patch16_224"  # fallback


def _x_positions(methods: list) -> dict:
    ordered = [m for m in METHOD_ORDER if m in methods]
    return {m: i for i, m in enumerate(ordered)}


def _legend_handles():
    handles = []
    for device, color in DEVICE_COLORS.items():
        handles.append(mpatches.Patch(color=color, label=f"Device: {device.upper()}"))
    handles.append(mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                                  markersize=8, label="FP32"))
    handles.append(mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                                  markersize=8, label="FP16"))
    for arch, size in ARCH_SIZES.items():
        handles.append(mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                                      markersize=np.sqrt(size) * 0.9,
                                      label=ARCH_LABELS[arch]))
    return handles


def _scatter(ax, row, x_pos: dict):
    m = row["pruning_method"]
    if m not in x_pos:
        return
    arch = row.get("architecture", "vit_tiny_patch16_224")
    x = x_pos[m] + ARCH_JITTER.get(arch, 0)
    color  = DEVICE_COLORS.get(row["device"], "gray")
    marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
    size   = ARCH_SIZES.get(arch, 80)

    y    = row["mean_latency_ms_mean"]
    yerr = row.get("mean_latency_ms_std", 0)
    ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                alpha=0.5, capsize=3, linewidth=1.0)
    ax.scatter(x, y, color=color, marker=marker, s=size,
               zorder=5, edgecolors="white", linewidths=0.4)


# ---------------------------------------------------------------------------
# Figure 1: Mean latency
# ---------------------------------------------------------------------------

def plot_latency(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 5), sharey=False)
    fig.suptitle("Single-Image Inference Latency by Compression Method (ViT)",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset]
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present)

        for _, row in sub.iterrows():
            _scatter(ax, row, x_pos)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in methods_present],
                           fontsize=8, rotation=20, ha="right")
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Mean Latency (ms)" if ax == axes[0] else "")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.16), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig1_latency_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 2: Throughput
# ---------------------------------------------------------------------------

def plot_throughput(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 5), sharey=False)
    fig.suptitle("Single-Image Throughput by Compression Method (ViT)",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset]
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            arch  = row.get("architecture", "vit_tiny_patch16_224")
            x     = x_pos[m] + ARCH_JITTER.get(arch, 0)
            color  = DEVICE_COLORS.get(row["device"], "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            size   = ARCH_SIZES.get(arch, 80)
            y      = row["throughput_imgs_per_s_mean"]
            yerr   = row.get("throughput_imgs_per_s_std", 0)
            ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                        alpha=0.5, capsize=3, linewidth=1.0)
            ax.scatter(x, y, color=color, marker=marker, s=size,
                       zorder=5, edgecolors="white", linewidths=0.4)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in methods_present],
                           fontsize=8, rotation=20, ha="right")
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Throughput (img/s)" if ax == axes[0] else "")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.16), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig2_throughput_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 3: Energy per image
# ---------------------------------------------------------------------------

def plot_energy(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 5), sharey=False)
    fig.suptitle("Mean Energy per Single Inference by Compression Method (ViT)",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset]
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            arch   = row.get("architecture", "vit_tiny_patch16_224")
            x      = x_pos[m] + ARCH_JITTER.get(arch, 0)
            color  = DEVICE_COLORS.get(row["device"], "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            size   = ARCH_SIZES.get(arch, 80)
            # energy (µWh) = (latency_ms / 1000) * power_W / 3600 * 1e6
            lat    = row["mean_latency_ms_mean"]
            pwr_col   = "mean_gpu_power_w_mean" if row["device"] == "cuda" else "mean_cpu_power_w_mean"
            s_pwr_col = "mean_gpu_power_w_std"  if row["device"] == "cuda" else "mean_cpu_power_w_std"
            pwr    = row[pwr_col]
            y      = (lat / 1000) * pwr / 3600 * 1e6
            # error propagation for product: σ_E/E = sqrt((σ_L/L)² + (σ_P/P)²)
            s_lat  = row.get("mean_latency_ms_std", 0) or 0
            s_pwr  = row.get(s_pwr_col, 0) or 0
            rel    = ((s_lat / lat) ** 2 + (s_pwr / pwr) ** 2) ** 0.5 if lat and pwr else 0
            yerr   = y * rel
            ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                        alpha=0.5, capsize=3, linewidth=1.0)
            ax.scatter(x, y, color=color, marker=marker, s=size,
                       zorder=5, edgecolors="white", linewidths=0.4)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in methods_present],
                           fontsize=8, rotation=20, ha="right")
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Energy (µWh / image)" if ax == axes[0] else "")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.16), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig3_energy_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 4: CPU slowdown factor (lat_cpu / lat_gpu)
# ---------------------------------------------------------------------------

def plot_slowdown(df: pd.DataFrame, output_dir: Path):
    cpu = df[df["device"] == "cpu"][
        ["dataset", "model_name", "pruning_method", "stored_precision",
         "architecture", "mean_latency_ms_mean"]
    ].rename(columns={"mean_latency_ms_mean": "lat_cpu"})

    gpu = df[df["device"] == "cuda"][
        ["dataset", "model_name", "pruning_method", "stored_precision",
         "architecture", "mean_latency_ms_mean"]
    ].rename(columns={"mean_latency_ms_mean": "lat_gpu"})

    merged = cpu.merge(gpu, on=["dataset", "model_name", "pruning_method",
                                 "stored_precision", "architecture"])
    merged["slowdown"] = merged["lat_cpu"] / merged["lat_gpu"]

    fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 5), sharey=True)
    fig.suptitle("CPU Slowdown Factor Relative to GPU (lat_cpu / lat_gpu) — ViT",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = merged[merged["dataset"] == dataset]
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            arch   = row.get("architecture", "vit_tiny_patch16_224")
            x      = x_pos[m] + ARCH_JITTER.get(arch, 0)
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            color  = "#1565c0" if row["stored_precision"] == "fp32" else "#e65100"
            size   = ARCH_SIZES.get(arch, 80)
            ax.scatter(x, row["slowdown"], color=color, marker=marker,
                       s=size, zorder=5, edgecolors="white", linewidths=0.4)

        ax.axhline(1, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels([METHOD_LABELS.get(m, m) for m in methods_present],
                           fontsize=8, rotation=20, ha="right")
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Slowdown Factor (×)" if ax == axes[0] else "")

    handles = [
        mlines.Line2D([], [], color="#1565c0", marker="^", linestyle="None",
                      markersize=8, label="FP32"),
        mlines.Line2D([], [], color="#e65100", marker="o", linestyle="None",
                      markersize=8, label="FP16"),
        mlines.Line2D([], [], color="gray", linestyle="--", linewidth=1,
                      label="No slowdown (×1)"),
    ] + [
        mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                      markersize=np.sqrt(s) * 0.9, label=ARCH_LABELS[a])
        for a, s in ARCH_SIZES.items()
    ]
    fig.legend(handles=handles, loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.16), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig4_cpu_slowdown.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 5: Model size (MB) vs mean latency
# ---------------------------------------------------------------------------

def plot_size_vs_latency(df: pd.DataFrame, output_dir: Path):
    method_colors = {
        "baseline":     "#455a64",
        "oneshot":      "#1565c0",
        "kd":           "#2e7d32",
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Model Size vs. Inference Latency (ViT)",
                 fontsize=14, fontweight="bold")

    for ax, device in zip(axes, ["cpu", "cuda"]):
        sub = df[df["device"] == device]
        for _, row in sub.iterrows():
            m      = row["pruning_method"]
            arch   = row.get("architecture", "vit_tiny_patch16_224")
            color  = method_colors.get(m, "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            size   = ARCH_SIZES.get(arch, 80)
            ax.scatter(row["model_size_mb"], row["mean_latency_ms_mean"],
                       color=color, marker=marker, s=size,
                       edgecolors="white", linewidths=0.4, zorder=5)

        ax.set_xlabel("Model Size (MB)")
        ax.set_ylabel("Mean Latency (ms)")
        ax.set_title(f"Device: {device.upper()}", fontweight="bold")

    method_handles = [
        mpatches.Patch(color=c, label=METHOD_LABELS.get(m, m))
        for m, c in method_colors.items()
    ]
    prec_handles = [
        mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                      markersize=8, label="FP32"),
        mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                      markersize=8, label="FP16"),
    ]
    arch_handles = [
        mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                      markersize=np.sqrt(s) * 0.9, label=ARCH_LABELS[a])
        for a, s in ARCH_SIZES.items()
    ]
    fig.legend(handles=method_handles + prec_handles + arch_handles,
               loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.16),
               frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig5_size_vs_latency.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Table 1: Summary averaged over datasets
# ---------------------------------------------------------------------------

def make_summary_table(df: pd.DataFrame, output_dir: Path):
    keep = ["pruning_method", "stored_precision", "architecture", "device",
            "mean_latency_ms_mean", "throughput_imgs_per_s_mean",
            "mean_cpu_power_w_mean", "mean_gpu_power_w_mean", "model_size_mb", "num_params"]

    sub = df[keep].copy()
    sub["_power"] = np.where(sub["device"] == "cuda", sub["mean_gpu_power_w_mean"], sub["mean_cpu_power_w_mean"])
    sub["energy_uwh"]    = ((sub["mean_latency_ms_mean"] / 1000) * sub["_power"] / 3600 * 1e6).round(3)
    sub["model_size_mb"] = sub["model_size_mb"].round(1)
    sub["num_params_M"]  = (sub["num_params"] / 1e6).round(2)

    agg = (
        sub.groupby(["pruning_method", "stored_precision", "architecture", "device"])
        .agg(
            latency_ms   =("mean_latency_ms_mean",          "mean"),
            throughput   =("throughput_imgs_per_s_mean",     "mean"),
            energy_uwh   =("energy_uwh",                     "mean"),
            model_size_mb=("model_size_mb",                  "first"),
            num_params_M =("num_params_M",                   "first"),
        )
        .reset_index()
    )

    agg["method_display"] = agg["pruning_method"].map(METHOD_LABELS).fillna(agg["pruning_method"])
    agg["arch_display"]   = agg["architecture"].map(ARCH_LABELS).fillna(agg["architecture"])
    agg = agg.sort_values(["pruning_method", "architecture", "stored_precision", "device"])
    agg["latency_ms"]  = agg["latency_ms"].round(2)
    agg["throughput"]  = agg["throughput"].round(1)
    agg["energy_uwh"]  = agg["energy_uwh"].round(3)

    display_cols = {
        "method_display":  "Method",
        "arch_display":    "Architecture",
        "stored_precision":"Precision",
        "device":          "Device",
        "num_params_M":    "Params (M)",
        "model_size_mb":   "Size (MB)",
        "latency_ms":      "Latency (ms)",
        "throughput":      "Throughput (img/s)",
        "energy_uwh":      "Energy (µWh/img)",
    }

    out_df = agg[list(display_cols.keys())].rename(columns=display_cols)
    out_path = output_dir / "table1_summary.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

    fig, ax = plt.subplots(figsize=(18, 0.4 * len(out_df) + 1.5))
    ax.axis("off")
    tbl = ax.table(cellText=out_df.values, colLabels=out_df.columns,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)
    tbl.auto_set_column_width(col=list(range(len(out_df.columns))))
    for j in range(len(out_df.columns)):
        tbl[(0, j)].set_facecolor("#37474f")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(out_df) + 1):
        bg = "#f5f5f5" if i % 2 == 0 else "white"
        for j in range(len(out_df.columns)):
            tbl[(i, j)].set_facecolor(bg)

    fig.suptitle("ViT Single-Image Benchmark Summary (averaged over 4 datasets)",
                 fontsize=10, fontweight="bold", y=0.98)
    plt.tight_layout()
    fig.savefig(output_dir / "table1_summary.png", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {output_dir / 'table1_summary.png'}")
    return out_df


# ---------------------------------------------------------------------------
# Table 2: Per-dataset breakdown
# ---------------------------------------------------------------------------

def make_per_dataset_table(df: pd.DataFrame, output_dir: Path):
    sub = df[["dataset", "pruning_method", "stored_precision", "architecture",
              "device", "mean_latency_ms_mean", "throughput_imgs_per_s_mean",
              "mean_cpu_power_w_mean", "mean_gpu_power_w_mean"]].copy()

    sub["_power"] = np.where(sub["device"] == "cuda", sub["mean_gpu_power_w_mean"], sub["mean_cpu_power_w_mean"])
    sub["energy_uwh"]      = ((sub["mean_latency_ms_mean"] / 1000) * sub["_power"] / 3600 * 1e6).round(3)
    sub["latency_ms"]      = sub["mean_latency_ms_mean"].round(2)
    sub["throughput"]      = sub["throughput_imgs_per_s_mean"].round(1)
    sub["method_display"]  = sub["pruning_method"].map(METHOD_LABELS).fillna(sub["pruning_method"])
    sub["arch_display"]    = sub["architecture"].map(ARCH_LABELS).fillna(sub["architecture"])

    out_df = sub[["dataset", "method_display", "arch_display", "stored_precision",
                  "device", "latency_ms", "throughput", "energy_uwh"]].rename(columns={
        "dataset":          "Dataset",
        "method_display":   "Method",
        "arch_display":     "Architecture",
        "stored_precision": "Precision",
        "device":           "Device",
        "latency_ms":       "Latency (ms)",
        "throughput":       "Throughput (img/s)",
        "energy_uwh":       "Energy (µWh/img)",
    })
    out_df = out_df.sort_values(["Dataset", "Method", "Architecture", "Precision", "Device"])

    out_path = output_dir / "table2_per_dataset.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return out_df

# ---------------------------------------------------------------------------
# Paths — edit these two lines, then run
# ---------------------------------------------------------------------------

INPUT_CSV  = Path("/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/ViT_averaged.csv")   # ← change this
OUTPUT_DIR = Path("/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/")          # ← change this

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(input_csv: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading: {input_csv}")
    df = load_data(input_csv)
    print(f"Rows after loading:  {len(df)}")
    print(f"Methods:      {sorted(df['pruning_method'].unique())}")
    print(f"Architectures:{sorted(df['architecture'].unique())}")
    print(f"Devices:      {sorted(df['device'].unique())}")
    print(f"Precisions:   {sorted(df['stored_precision'].unique())}")
    print()

    plot_latency(df, output_dir)
    plot_throughput(df, output_dir)
    plot_energy(df, output_dir)
    plot_slowdown(df, output_dir)
    plot_size_vs_latency(df, output_dir)
    make_summary_table(df, output_dir)
    make_per_dataset_table(df, output_dir)

    print("\nAll outputs written to:", output_dir)


main(INPUT_CSV, OUTPUT_DIR)



Loading: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/ViT_averaged.csv
Rows after loading:  72
Methods:      ['baseline', 'kd', 'oneshot']
Architectures:['vit_base_patch16_224', 'vit_small_patch16_224', 'vit_tiny_patch16_224']
Devices:      ['cpu', 'cuda']
Precisions:   ['fp32']

Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/fig1_latency_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/fig2_throughput_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/fig3_energy_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/fig4_cpu_slowdown.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/rerun/ViT/fig5_size_vs_latency.png
Saved: /Users/arihangupta/Downloads/pruning_pro